# 0. Setup & Installation

In [2]:
!pip install -q langchain langchain-community chromadb sentence-transformers google-generativeai tiktoken langchain-text-splitters

In [4]:
import os
import json
from typing import List, Dict
import google.generativeai as genai
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.utils import embedding_functions
import numpy as np

/var/folders/s1/sxxd1mx915d_tx5shvddn3_80000gn/T/ipykernel_16049/702642590.py:4: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [5]:
!pip install openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 35.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [openai]2m1/2 [openai]


In [6]:
from google.colab import userdata
from openai import OpenAI

# Get OpenAI API key
try:
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
except:
    OPENAI_API_KEY = input('Enter your OpenAI API Key: ')

# Initialize OpenAI client
client = OpenAI(api_key=OPENAI_API_KEY)

print("✅ OpenAI API configured!")

# Example test call
response = client.responses.create(
    model="gpt-4.1-mini",
    input="Hello! Can you confirm the API is working?"
)

print(response.output_text)

ModuleNotFoundError: No module named 'google.colab'

# 1. Domain Selection

In [ ]:
#upon professor approval, sklearn fetch_20newsgroups is used as domain
import sklearn

from sklearn import datasets

In [ ]:
domain = sklearn.datasets.fetch_20newsgroups(subset='train', categories=['sci.med'], remove=('footers', 'quotes'),)

# 2. Document Processing

In [ ]:
dom_meta = []
dom_data = []

for doc in domain.data[:100]:
    parts = doc.split('\n\n', 1)  # split header and body
    if len(parts) == 2:
        header, body = parts
    else:
        header = parts[0]
        body = ""

    dom_meta.append(header)
    dom_data.append(body)

In [ ]:
from email.parser import Parser

dom_meta_parsed = []

for header in dom_meta:
    parsed = Parser().parsestr(header)
    dom_meta_parsed.append(dict(parsed.items()))

In [ ]:
def recursive_chunk(text, size=500, overlap=50):
    '''Recursive chunking respecting boundaries.'''
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=size,
        chunk_overlap=overlap,
        separators=['\n\n', '\n', '. ', ' ', '']
    )
    return splitter.split_text(text)

In [ ]:
def chunk_with_metadata(text, meta, chunk_size=400):
    '''Chunk documents preserving metadata.'''
    all_chunks = []
    for i in range(len(text)):
        chunks = recursive_chunk(text[i], chunk_size)
        for j, chunk in enumerate(chunks):
            all_chunks.append({
                'text': chunk,
                'metadata': {
                    'From': meta[i].get('From', ''),
                    'Subject': meta[i].get('Subject', ''),
                    'Organization': meta[i].get('Organization', ''),
                    'Lines': meta[i].get('Lines', ''),
                    'doc_id': i,
                    'chunk_idx': j
                }
            })
    return all_chunks

In [ ]:
chunked_dom = chunk_with_metadata(dom_data, dom_meta_parsed)

# 3. Vector Database

In [ ]:
# Initialize ChromaDB
ef = embedding_functions.SentenceTransformerEmbeddingFunction('all-MiniLM-L6-v2')
chroma_client = chromadb.PersistentClient(path='./chroma_db') # Ensure PersistentClient is used

med_col = chroma_client.get_or_create_collection('medical', embedding_function=ef) # Use get_or_create_collection

# Add chunks
def add_chunks(col, chunks):
    col.add(
        documents=[c['text'] for c in chunks],
        metadatas=[c['metadata'] for c in chunks],
        ids=[f"{c['metadata']['doc_id']}_{c['metadata']['chunk_idx']}" for c in chunks]
    )

In [ ]:
add_chunks(med_col, chunked_dom)

# 4. RAG Implementation

In [ ]:
def retrieve(col, query, top_k=5):
    '''Retrieve relevant chunks.'''
    results = col.query(query_texts=[query], n_results=top_k,
                        include=['documents', 'metadatas', 'distances'])
    return [{
        'text': results['documents'][0][i],
        'metadata': results['metadatas'][0][i],
        'similarity': 1 - results['distances'][0][i]
    } for i in range(len(results['documents'][0]))]

# Test
query = 'Where can I find a comprehensive review?'
results = retrieve(med_col, query)
print(f'Query: {query}\n')
for r in results:
    print(f"📄 {r['metadata']['doc_id']} (sim: {r['similarity']:.3f})")

In [ ]:
client = OpenAI(api_key=OPENAI_API_KEY)

def rag_query(query, col, top_k=5):
    """Complete RAG pipeline using OpenAI model."""

    # 🔎 Retrieve
    chunks = retrieve(col, query, top_k)

    # 🧠 Build context
    context = '\n---\n'.join([
        f"[Source {i+1}: {c['metadata']['doc_id']}]\n{c['text']}"
        for i, c in enumerate(chunks)
    ])

    prompt = f"""Answer using ONLY the context below. Cite sources as [Source X].
If not in context, say "I don't have that information."

CONTEXT:
{context}

QUESTION: {query}

ANSWER:"""

    # 🤖 OpenAI Chat Completion
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )

    answer = response.choices[0].message.content

    return {
        "answer": answer,
        "sources": [c["metadata"] for c in chunks]
    }

In [ ]:
# Test rag_query
result = rag_query("Where can I find a comprehensive review?", med_col)

print("🤖 RAG Response:")
print(result["answer"])

# 5. Conversation Memory

In [ ]:
# Memory buffer for the last ten exchanges
from collections import deque

# stores alternating user/assistant messages
CHAT_MEMORY = deque(maxlen=20)  # 10 exchanges = 20 messages

In [ ]:
# Helper formatting
def format_history_for_prompt(history_deque) -> str:
    if not history_deque:
        return "(no prior conversation)"
    lines = []
    for m in history_deque:
        role = "User" if m["role"] == "user" else "Assistant"
        lines.append(f"{role}: {m['content']}")
    return "\n".join(lines)

In [ ]:
# Follow-up handling with query rewriting
def rewrite_to_standalone_query(user_query: str, history_deque) -> str:
    """
    Uses the model to rewrite a follow-up into a standalone query for retrieval.
    If there's no history, returns the original query.
    """
    if not history_deque:
        return user_query

    history_text = format_history_for_prompt(history_deque)

    rewrite_prompt = f"""
You will be given a conversation and the user's latest question.
Rewrite the latest question into a standalone search query that includes necessary context.
- Keep it short (1 sentence).
- If it's already standalone, return it unchanged.
- Do NOT answer the question.

CONVERSATION:
{history_text}

LATEST QUESTION:
{user_query}

STANDALONE SEARCH QUERY:
""".strip()

    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You rewrite questions for retrieval."},
            {"role": "user", "content": rewrite_prompt}
        ],
        temperature=0
    )
    return resp.choices[0].message.content.strip()

In [ ]:
# Updated RAG query with memory and rewritten retrieval queries!
def rag_query_with_memory(user_query: str, col, top_k=5, memory=CHAT_MEMORY):
    """RAG + conversation memory (last 5–10 exchanges) + follow-up handling."""

    # 1) Rewrite for retrieval (follow-up support)
    standalone_query = rewrite_to_standalone_query(user_query, memory)

    # 2) Retrieve using standalone query
    chunks = retrieve_with_rerank(col, standalone_query, initial_k=20, final_k=top_k)

    context = '\n---\n'.join([
        f"[Source {i+1}: doc {c['metadata']['doc_id']}, chunk {c['metadata']['chunk_idx']}]\n{c['text']}"
        for i, c in enumerate(chunks)
    ]) or "(no retrieved context)"

    # 3) Build conversation context for the answer prompt
    history_text = format_history_for_prompt(memory)

    prompt = f"""You are a RAG assistant with conversation memory.

Rules:
- Use ONLY the CONTEXT for factual claims about the domain.
- You may use the CONVERSATION HISTORY to resolve references (it/that/they) and user preferences.
- If the answer is not in CONTEXT, say: "I don't have that information."
- Cite sources like [Source 1], [Source 2].

CONVERSATION HISTORY (last turns):
{history_text}

CONTEXT:
{context}

USER QUESTION:
{user_query}

ANSWER:
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )

    answer = response.choices[0].message.content.strip()

    # 4) Update memory (store the actual user query + assistant answer)
    memory.append({"role": "user", "content": user_query})
    memory.append({"role": "assistant", "content": answer})

    return {
        "answer": answer,
        "sources": [c["metadata"] for c in chunks],
        "standalone_query_used_for_retrieval": standalone_query
    }

In [ ]:
# # Test
# res1 = rag_query_with_memory("Where can I find a comprehensive review?", med_col)
# print(res1["answer"])
# print("Standalone:", res1["standalone_query_used_for_retrieval"])

# res2 = rag_query_with_memory("Are you sure about that?", med_col)
# print(res2["answer"])
# print("Standalone:", res2["standalone_query_used_for_retrieval"])

# 6. Advanced Feature

In [ ]:
!pip install -q sentence-transformers

In [ ]:
from sentence_transformers import CrossEncoder

In [ ]:
# Lightweight reranker model (good tradeoff of quality/speed)
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

# Create a rerank function
def rerank_chunks(query: str, chunks: list, top_n: int = 5):
    """
    chunks: list of dicts like your retrieve() returns:
      { 'text': ..., 'metadata': ..., 'similarity': ... }
    """
    if not chunks:
        return []

    pairs = [(query, c["text"]) for c in chunks]
    scores = reranker.predict(pairs)  # higher = better

    # attach rerank score
    for c, s in zip(chunks, scores):
        c["rerank_score"] = float(s)

    # sort by rerank_score desc
    chunks = sorted(chunks, key=lambda x: x["rerank_score"], reverse=True)
    return chunks[:top_n]

In [ ]:
# Modify the retrieval to do: top-20 -> rerank -> top-5
def retrieve_with_rerank(col, query: str, initial_k: int = 20, final_k: int = 5):
    # 1) initial semantic retrieve (top-20)
    print("Using reranking — top 20 → top", final_k)
    results = col.query(
        query_texts=[query],
        n_results=initial_k,
        include=["documents", "metadatas", "distances"]
    )

    chunks = []
    for i in range(len(results["documents"][0])):
        chunks.append({
            "text": results["documents"][0][i],
            "metadata": results["metadatas"][0][i],
            "similarity": 1 - results["distances"][0][i],  # your approach
        })

    # 2) rerank to top-5
    return rerank_chunks(query, chunks, top_n=final_k)

In [ ]:
#re-test after adding advanced feature
# Reset memory before test
CHAT_MEMORY.clear()

print("=== TEST AFTER STEP 6 INTEGRATION ===")

res1 = rag_query_with_memory(
    "Where can I find a comprehensive review?",
    med_col
)

print("\nAnswer 1:")
print(res1["answer"])
print("Standalone query:", res1["standalone_query_used_for_retrieval"])

res2 = rag_query_with_memory(
    "Are you sure about that?",
    med_col
)

print("\nAnswer 2:")
print(res2["answer"])
print("Standalone query:", res2["standalone_query_used_for_retrieval"])

# 7. Deployment (Public URL + UI + citations)

In [ ]:
!pip install -q gradio
import gradio as gr
from collections import deque

def new_memory():
    return deque(maxlen=20)  # 10 exchanges

def respond(user_message, ui_history, memory_state):
    try:
        if ui_history is None:
            ui_history = []
        if memory_state is None:
            memory_state = new_memory()

        # Use your rag function with per-session memory
        result = rag_query_with_memory(user_message, med_col, memory=memory_state)

        answer = result["answer"]
        sources = result.get("sources", [])

        if sources:
            citation_text = "\n".join([
                f"[Source {i+1}] doc={s.get('doc_id')} chunk={s.get('chunk_id')} subject={s.get('Subject','')}"
                for i, s in enumerate(sources)
            ])
        else:
            citation_text = "No sources retrieved."

        final_answer = f"{answer}\n\n---\nSources:\n{citation_text}"

        ui_history = ui_history + [[user_message, final_answer]]
        return ui_history, ui_history, memory_state

    except Exception as e:
        ui_history = (ui_history or []) + [[user_message, f"⚠️ Error: {repr(e)}"]]
        return ui_history, ui_history, memory_state

with gr.Blocks() as demo:
    gr.Markdown("# RAG Chatbot (Chroma + Memory + Reranking)")
    chatbot = gr.Chatbot()
    state_ui = gr.State([])
    state_mem = gr.State(new_memory())

    msg = gr.Textbox(label="Your message")
    send = gr.Button("Send")
    clear = gr.Button("Clear")

    send.click(respond, inputs=[msg, state_ui, state_mem], outputs=[chatbot, state_ui, state_mem])
    msg.submit(respond, inputs=[msg, state_ui, state_mem], outputs=[chatbot, state_ui, state_mem])

    def do_clear():
        return [], [], new_memory()

    clear.click(do_clear, outputs=[chatbot, state_ui, state_mem])

demo.launch(share=True, debug=True)